# Processing Global-Scale Datasets (CHIRPS, ERA5, TerraClimate)

This notebook demonstrates how to calculate **SPI** and **SPEI** for large global datasets that exceed available RAM.

**Key Features:**
- Memory-efficient chunked processing with spatial tiling
- Automatic memory estimation and chunk size optimization
- Multi-distribution support (Gamma, Pearson III, Log-Logistic)
- Progress tracking for long-running computations
- Streaming I/O to avoid loading entire datasets

**Common Global Datasets:**

| Dataset | Resolution | Grid Size | Approx. Size (monthly, 40+ yr) |
|---------|-----------|-----------|-------------------------------|
| CHIRPS v3 | 0.05 deg | 2400 x 7200 | ~69 GB |
| ERA5 | 0.25 deg | 721 x 1440 | ~4 GB |
| CPC Global | 0.5 deg | 360 x 720 | ~0.7 GB |
| TerraClimate | ~4 km | 4320 x 8640 | ~130 GB |

**Processing Methods:**

| Method | Best For | Section |
|--------|----------|---------|
| **Option A** --- Simple Global | Most users, quick setup | 4.1 / 5.1 |
| **Option B** --- Chunked Processor | Full control, progress tracking, parameter saving | 4.2 / 5.2 |
| **Option C** --- Dask Lazy | Very large data, Zarr output | 4.3 / 5.3 |

## 1. Setup and Imports

In [1]:
# Add src directory to Python path
import sys
sys.path.insert(0, '../src')

# Core libraries
import numpy as np
import xarray as xr
from pathlib import Path

# Import chunked processing functions
from chunked import (
    ChunkedProcessor,
    estimate_memory,
    estimate_memory_from_data,
    MemoryEstimate,
    compute_spi_global,
    compute_spei_global
)

# Import standard functions (for comparison / small-data fallback)
from indices import (
    spi,
    spei,
    spi_global,
    spei_global,
    estimate_memory_requirements,
    save_fitting_params,
    load_fitting_params,
    save_index_to_netcdf
)

from compute import compute_index_dask, compute_index_dask_to_zarr
from utils import get_optimal_chunk_size, print_memory_info
from config import Periodicity

print("All imports successful!")

All imports successful!


## 2. Configuration

Set paths and parameters for your global data. Update `BASE_DIR` to point to your data directory.

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================
# Set your base data directory. Works with Windows, WSL, or Linux:
#   Windows:  r"C:\Users\benny\data\global"
#   WSL:      "/mnt/c/Users/benny/data/global"
#   Linux:    "/home/user/data/global"

BASE_DIR = r"C:\Users\benny\OneDrive\Documents\Github\precip-index\global"  # <-- CHANGE THIS

# ============================================================
# DERIVED PATHS (no changes needed below)
# ============================================================
base_dir      = Path(BASE_DIR)
input_dir     = base_dir / 'input'
output_dir    = base_dir / 'output'
output_netcdf = output_dir / 'netcdf'
output_params = output_dir / 'params'
output_plots  = output_dir / 'plots'

# Create output directories
for d in [output_netcdf, output_params, output_plots]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# INPUT FILES  (per-year TerraClimate global files)
# ============================================================
# SPI: precipitation only
# Use a glob pattern — open_nc() / compute_spi_global() will merge all matching files.
chirps_file     = 'wld_cli_chirps3_month1_1981_2024_d1_mod.nc'   # single merged CHIRPS file
tc_precip_glob  = str(input_dir / 'TerraClimate_ppt_*.nc')       # per-year TerraClimate ppt
tc_pet_glob     = str(input_dir / 'TerraClimate_pet_*.nc')       # per-year TerraClimate pet

# ============================================================
# CALIBRATION PERIOD (WMO standard: 1991-2020)
# ============================================================
calib_start = 1991
calib_end   = 2020

# ============================================================
# GPU ACCELERATION
# ============================================================
# None = auto-detect CuPy; True = require GPU; False = CPU only
use_gpu = None

from gpu import gpu_info
print(f"Base directory:   {base_dir}")
print(f"Input directory:  {input_dir}")
print(f"Output directory: {output_dir}")
print(f"Calibration:      {calib_start}-{calib_end}")
print(f"\nSPI input (CHIRPS): {chirps_file}")
print(f"SPEI precip (glob): {tc_precip_glob}")
print(f"SPEI PET (glob):    {tc_pet_glob}")
print(f"\n{gpu_info()}")


Base directory:   C:\Users\benny\OneDrive\Documents\Github\precip-index\global
Input directory:  C:\Users\benny\OneDrive\Documents\Github\precip-index\global\input
Output directory: C:\Users\benny\OneDrive\Documents\Github\precip-index\global\output
Calibration:      1991-2020

SPI input:  wld_cli_chirps3_month1_1981_2024_d1_mod.nc
SPEI precip: wld_cli_terraclimate_ppt_1958_2024.nc
SPEI PET:    wld_cli_terraclimate_pet_1958_2024.nc


## 3. Memory Estimation & Chunk Size

Before processing, always estimate memory requirements to determine if chunking is needed.
The `estimate_memory()` function calculates peak memory usage and recommends chunk sizes.

### 3.1 Estimate Memory Requirements

In [3]:
# Estimate memory for CHIRPS v3 (0.05 deg, Feb 1981 - Dec 2025)
mem_chirps = estimate_memory(
    n_time=539,    # Feb 1981 - Dec 2025 = 539 months
    n_lat=2400,    # CHIRPS v3 global
    n_lon=7200
)
print("CHIRPS v3 (0.05 deg global):")
print(mem_chirps)

print("\n" + "=" * 60)

# Estimate memory for TerraClimate (~4 km, 1958-2024)
mem_tc = estimate_memory(
    n_time=804,    # 67 years * 12 months
    n_lat=4320,    # TerraClimate global
    n_lon=8640
)
print("\nTerraClimate (~4 km global):")
print(mem_tc)

print("\n" + "=" * 60)

# Current system memory
print_memory_info()

2026-01-28 15:57:49 | INFO     | utils | Memory: 37.56 GB used / 382.60 GB total (9.8% used)


CHIRPS v3 (0.05 deg global):
MemoryEstimate(
  Input size: 69.39 GB
  Peak memory needed: 832.73 GB
  Available memory: 345.05 GB
  Status: ✗ Requires chunking
  Recommended chunk size: (2238, 2238) (lat, lon)
  Number of chunks: 8
)


TerraClimate (~4 km global):
MemoryEstimate(
  Input size: 223.59 GB
  Peak memory needed: 2683.03 GB
  Available memory: 345.04 GB
  Status: ✗ Requires chunking
  Recommended chunk size: (1833, 1833) (lat, lon)
  Number of chunks: 15
)



### 3.2 Chunk Size Guidelines

Choose chunk size based on your available RAM:

| Available RAM | Recommended Chunk Size | Notes |
|--------------|------------------------|-------|
| 16 GB | 200 x 200 | Minimum recommended |
| 32 GB | 300 x 300 | Good for workstations |
| 64 GB | 500 x 500 | Standard servers |
| 128 GB | 800 x 800 | High-memory systems |
| 256 GB+ | 1200 x 1200 | Large servers |

**Tips:**
- Larger chunks = faster (fewer I/O operations)
- Smaller chunks = lower memory usage
- Start conservative and increase if stable
- The `get_optimal_chunk_size()` function automates this decision

In [5]:
# Calculate optimal chunk size for different scenarios
print("Optimal chunk sizes for common datasets:")
print("=" * 60)

datasets = [
    ("CHIRPS v3 0.05 deg", 539, 2400, 7200),
    ("ERA5 0.25 deg",      528, 721,  1440),
    ("CPC 0.5 deg",        528, 360,  720),
    ("TerraClimate ~4 km", 804, 4320, 8640),
]

for name, nt, nlat, nlon in datasets:
    chunk_lat, chunk_lon = get_optimal_chunk_size(
        n_time=nt,
        n_lat=nlat,
        n_lon=nlon,
        available_memory_gb=256.0  # Adjust to your system
    )
    print(f"\n  {name}:")
    print(f"    Grid: {nlat} x {nlon}, Time steps: {nt}")
    print(f"    Recommended chunk: ({chunk_lat}, {chunk_lon})")

Optimal chunk sizes for common datasets:

  CHIRPS v3 0.05 deg:
    Grid: 2400 x 7200, Time steps: 539
    Recommended chunk: (1928, 1928)

  ERA5 0.25 deg:
    Grid: 721 x 1440, Time steps: 528
    Recommended chunk: (721, 1440)

  CPC 0.5 deg:
    Grid: 360 x 720, Time steps: 528
    Recommended chunk: (360, 720)

  TerraClimate ~4 km:
    Grid: 4320 x 8640, Time steps: 804
    Recommended chunk: (1578, 1578)


### 3.3 ChunkedProcessor & Progress Callback

The `ChunkedProcessor` is used by Options B throughout this notebook. Configure it once here and reuse for all SPI/SPEI runs.

In [6]:
# Create processor with custom settings
processor = ChunkedProcessor(
    chunk_lat=1200,   # Process 1200 latitude rows at a time
    chunk_lon=1200,   # Process 1200 longitude columns at a time
    n_workers=20,     # Number of parallel workers (adjust to your CPU)
    verbose=True,     # Print progress information
    use_gpu=use_gpu   # GPU acceleration (configured in cell 4)
)

print(f'Processor configured:')
print(f'  Chunk size: {processor.chunk_lat} x {processor.chunk_lon}')
print(f'  Workers:    {processor.n_workers}')
print(f'  Temp dir:   {processor.temp_dir}')


2026-01-28 15:58:59 | INFO     | chunked | ChunkedProcessor initialized: chunk_size=(1200, 1200), workers=20


Processor configured:
  Chunk size: 1200 x 1200
  Workers:    20
  Temp dir:   C:\TEMP


In [7]:
# Define a progress callback (optional)
def progress_callback(chunk_info, progress_pct):
    """Called after each chunk is processed."""
    if progress_pct % 10 < 2:  # Print every ~10%
        print(f"  [{progress_pct:5.1f}%] {chunk_info}")

## 4. SPI Global Processing

SPI requires only **precipitation** data. This section uses CHIRPS v3 global monthly data.

Three processing methods are available --- choose based on your needs:

| Method | Pros | Cons |
|--------|------|------|
| **Option A**: `compute_spi_global()` | Simplest API, minimal code | Less control |
| **Option B**: `ChunkedProcessor` | Full control, progress tracking, param saving | More setup |
| **Option C**: Dask | Lazy evaluation, Zarr output | Requires Dask |

### 4.1 Option A: Simple Global SPI

In [ ]:
# ============================================================
# Option A: Simple Global SPI
# ============================================================
# Minimal setup --- just provide file path or glob pattern.
# Variable name is auto-detected from the file.
# Uncomment to run:

# --- CHIRPS (single merged file) ---
# result = compute_spi_global(
#     precip_path=str(input_dir / chirps_file),
#     output_path=str(output_netcdf / 'wld_cli_chirps3_d2_spi_gamma_12_month.nc'),
#     scale=12,
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     chunk_size=1200,
#     distribution='gamma',
#     use_gpu=use_gpu
# )

# --- TerraClimate (per-year files via glob) ---
# result = compute_spi_global(
#     precip_path=tc_precip_glob,  # 'input/TerraClimate_ppt_*.nc'
#     output_path=str(output_netcdf / 'wld_cli_tc_spi_gamma_12_month.nc'),
#     scale=12,
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     chunk_size=1200,
#     distribution='gamma',
#     use_gpu=use_gpu
# )


### 4.2 Option B: Advanced Chunked SPI

In [8]:
# ============================================================
# Option B: Chunked SPI with full control
# ============================================================
# Uses the ChunkedProcessor configured in Section 3.3.
# Uncomment to run:

result = processor.compute_spi_chunked(
    precip=str(input_dir / chirps_file),
    output_path=str(output_netcdf / 'wld_cli_chirps3_d1_spi_gamma_12_month.nc'),
    scale=12,
    periodicity='monthly',
    calibration_start_year=calib_start,
    calibration_end_year=calib_end,
    distribution='gamma',         # 'gamma', 'pearson3', or 'log_logistic'
    save_params=True,             # Save fitting parameters for reuse
    params_path=str(output_params / 'wld_cli_chirps3_d1_spi_gamma_12_month_params.nc'),
    compress=True,
    complevel=4,
    callback=progress_callback
)


2026-01-28 15:59:24 | INFO     | chunked | Opening data: C:\Users\benny\OneDrive\Documents\Github\precip-index\global\input\wld_cli_chirps3_month1_1981_2024_d1_mod.nc
C:\Users\benny\OneDrive\Documents\Github\precip-index\notebook\../src\chunked.py:337: UserWarning: The specified chunks separate the stored chunks along dimension "lon" starting at index 1200. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(precip, chunks={'time': -1, 'lat': self.chunk_lat, 'lon': self.chunk_lon})
2026-01-28 15:59:25 | INFO     | chunked | 
MemoryEstimate(
  Input size: 69.39 GB
  Peak memory needed: 832.73 GB
  Available memory: 344.96 GB
  Status: ✗ Requires chunking
  Recommended chunk size: (2238, 2238) (lat, lon)
  Number of chunks: 8
)
2026-01-28 15:59:25 | INFO     | chunked | Creating output file: C:\Users\benny\OneDrive\Documents\Github\precip-index\global\output\netcdf\wld_cli_chirps3_d1_spi_gamma_12_month.nc
2026-01-28 16:04:34 | INFO     | chu

  [ 41.7%] Chunk 5/12 [lat 0:1200, lon 4800:6000]


2026-01-28 17:02:23 | INFO     | compute | Computing index: shape=(539, 1200, 1200), scale=12, distribution=gamma, grid_cells=1,440,000
2026-01-28 17:02:23 | INFO     | compute | Data has 539 timesteps (not divisible by 12). Padding 1 NaN timesteps to complete final year (45 years total).
2026-01-28 17:02:23 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-01-28 17:02:43 | INFO     | compute | Step 2/3: Computing gamma parameters...
2026-01-28 17:02:58 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-01-28 17:04:45 | INFO     | compute | Index computation complete
2026-01-28 17:14:14 | INFO     | chunked | Processing Chunk 7/12 [lat 1200:2400, lon 0:1200] (58.3%)


  [ 50.0%] Chunk 6/12 [lat 0:1200, lon 6000:7200]


2026-01-28 17:14:40 | INFO     | compute | Computing index: shape=(539, 1200, 1200), scale=12, distribution=gamma, grid_cells=1,440,000
2026-01-28 17:14:40 | INFO     | compute | Data has 539 timesteps (not divisible by 12). Padding 1 NaN timesteps to complete final year (45 years total).
2026-01-28 17:14:40 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-01-28 17:15:00 | INFO     | compute | Step 2/3: Computing gamma parameters...
2026-01-28 17:15:13 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-01-28 17:16:32 | INFO     | compute | Index computation complete
2026-01-28 17:26:09 | INFO     | chunked | Processing Chunk 8/12 [lat 1200:2400, lon 1200:2400] (66.7%)
2026-01-28 17:26:35 | INFO     | compute | Computing index: shape=(539, 1200, 1200), scale=12, distribution=gamma, grid_cells=1,440,000
2026-01-28 17:26:35 | INFO     | compute | Data has 539 timesteps (not divisible by 12). Padding 1 NaN timesteps to complete final year (45 years t

  [ 91.7%] Chunk 11/12 [lat 1200:2400, lon 4800:6000]


2026-01-28 18:28:24 | INFO     | compute | Computing index: shape=(539, 1200, 1200), scale=12, distribution=gamma, grid_cells=1,440,000
2026-01-28 18:28:24 | INFO     | compute | Data has 539 timesteps (not divisible by 12). Padding 1 NaN timesteps to complete final year (45 years total).
2026-01-28 18:28:24 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-01-28 18:28:44 | INFO     | compute | Step 2/3: Computing gamma parameters...
2026-01-28 18:28:58 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-01-28 18:30:42 | INFO     | compute | Index computation complete
2026-01-28 18:44:30 | INFO     | chunked | Saving fitting parameters to: C:\Users\benny\OneDrive\Documents\Github\precip-index\global\output\params\wld_cli_chirps3_d1_spi_gamma_12_month_params.nc
2026-01-28 18:44:30 | INFO     | indices | Saving gamma fitting parameters to: C:\Users\benny\OneDrive\Documents\Github\precip-index\global\output\params\wld_cli_chirps3_d1_spi_gamma_12_month

  [100.0%] Chunk 12/12 [lat 1200:2400, lon 6000:7200]


2026-01-28 18:44:58 | INFO     | indices | Fitting parameters saved: C:\Users\benny\OneDrive\Documents\Github\precip-index\global\output\params\wld_cli_chirps3_d1_spi_gamma_12_month_params.nc
2026-01-28 18:44:58 | INFO     | chunked | Chunked SPI computation complete: C:\Users\benny\OneDrive\Documents\Github\precip-index\global\output\netcdf\wld_cli_chirps3_d1_spi_gamma_12_month.nc


### 4.3 Option C: Dask-based Lazy SPI

In [ ]:
# ============================================================
# Option C: Dask lazy SPI
# ============================================================
# Builds a computation graph without executing --- results are
# computed only when you write to disk.
# Uncomment to run:

# ds = xr.open_dataset(
#     str(input_dir / chirps_file),
#     chunks={'time': -1, 'lat': 1200, 'lon': 1200}
# )
# precip = ds['precip']
#
# # Build computation graph (lazy - no computation yet)
# spi_lazy, _ = compute_index_dask(
#     precip,
#     scale=12,
#     data_start_year=1981,
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     periodicity=Periodicity.monthly,
#     distribution='gamma'
# )
#
# # Execute and stream directly to NetCDF
# spi_lazy.to_netcdf(str(output_netcdf / 'wld_cli_chirps3_d2_spi_gamma_12_month_dask.nc'))

# --- Or use Zarr for better streaming performance ---
# compute_index_dask_to_zarr(
#     precip,
#     str(output_netcdf / 'spi_12_global.zarr'),
#     scale=12,
#     data_start_year=1981,
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     periodicity=Periodicity.monthly,
#     n_workers=8
# )

print("Option C: Dask lazy SPI (uncomment to run)")

### 4.4 Reusing SPI Parameters

For repeated calculations with the same calibration period (e.g., updating with new months of data),
load previously saved parameters to skip the fitting step.

In [ ]:
# Step 1: First run --- save parameters using save_params=True (any method above)
# The chunked processor automatically saves params when save_params=True.

# Step 2: Load previously saved parameters
# params = load_fitting_params(
#     str(output_params / 'wld_cli_chirps3_d2_spi_gamma_12_month_params.nc'),
#     scale=12,
#     periodicity='monthly'
# )
# print(f"Loaded parameters: {list(params.keys())}")
# for k, v in params.items():
#     if hasattr(v, 'shape'):
#         print(f"  {k}: shape={v.shape}")

# Step 3: Re-run with pre-computed parameters (much faster)
# result_fast = processor.compute_spi_chunked(
#     precip=str(input_dir / 'updated_precip_data.nc'),
#     output_path=str(output_netcdf / 'spi_12_updated.nc'),
#     scale=12,
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     distribution='gamma',
#     save_params=False  # Already have params
# )

print("Parameter reuse workflow (uncomment to run)")

## 5. SPEI Global Processing

SPEI requires both **precipitation** and **PET** (Potential Evapotranspiration) data.
This section uses TerraClimate global monthly data (1958--2024).

**Input files** (in the same `input` directory):

| File | Variable | Period |
|:-----|:---------|:-------|
| `wld_cli_terraclimate_ppt_1958_2024.nc` | `ppt` | 1958--2024 |
| `wld_cli_terraclimate_pet_1958_2024.nc` | `pet` | 1958--2024 |

Variable names (`ppt`, `pet`) are auto-detected by the processor.

### 5.1 Option A: Simple Global SPEI

In [ ]:
# ============================================================
# Option A: Simple Global SPEI
# ============================================================
# Minimal setup --- just provide precip and PET file paths or globs.
# Uncomment to run:

# --- SPEI-12 with Gamma distribution (TerraClimate per-year files) ---
# result_spei = compute_spei_global(
#     precip_path=tc_precip_glob,   # 'input/TerraClimate_ppt_*.nc'
#     pet_path=tc_pet_glob,         # 'input/TerraClimate_pet_*.nc'
#     output_path=str(output_netcdf / 'wld_cli_tc_spei_gamma_12_month.nc'),
#     scale=12,
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     chunk_size=1200,
#     distribution='gamma',
#     use_gpu=use_gpu
# )

# --- SPEI-12 with Pearson III (recommended for SPEI) ---
# result_spei = compute_spei_global(
#     precip_path=tc_precip_glob,
#     pet_path=tc_pet_glob,
#     output_path=str(output_netcdf / 'wld_cli_tc_spei_pearson3_12_month.nc'),
#     scale=12,
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     chunk_size=1200,
#     distribution='pearson3',
#     use_gpu=use_gpu
# )


### 5.2 Option B: Advanced Chunked SPEI

Uses the same `ChunkedProcessor` configured in Section 3.3.

In [9]:
# ============================================================
# Option B: Chunked SPEI with full control
# ============================================================
# Uncomment to run:

# --- SPEI-12 with Gamma distribution ---
# result_spei = processor.compute_spei_chunked(
#     precip=str(input_dir / tc_precip_file),
#     pet=str(input_dir / tc_pet_file),
#     output_path=str(output_netcdf / 'wld_cli_terraclimate_spei_gamma_12_month.nc'),
#     scale=12,
#     periodicity='monthly',
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     distribution='gamma',
#     save_params=True,
#     params_path=str(output_params / 'wld_cli_terraclimate_spei_gamma_12_month_params.nc'),
#     compress=True,
#     complevel=4,
#     callback=progress_callback
# )

# --- SPEI-12 with Pearson III distribution ---
result_spei_p3 = processor.compute_spei_chunked(
    precip=str(input_dir / tc_precip_file),
    pet=str(input_dir / tc_pet_file),
    output_path=str(output_netcdf / 'wld_cli_terraclimate_spei_pearson3_12_month.nc'),
    scale=12,
    periodicity='monthly',
    calibration_start_year=calib_start,
    calibration_end_year=calib_end,
    distribution='pearson3',
    save_params=True,
    params_path=str(output_params / 'wld_cli_terraclimate_spei_pearson3_12_month_params.nc'),
    compress=True,
    complevel=4,
    callback=progress_callback
)

2026-01-28 18:45:00 | INFO     | chunked | Opening precipitation data
C:\Users\benny\OneDrive\Documents\Github\precip-index\notebook\../src\chunked.py:629: UserWarning: The specified chunks separate the stored chunks along dimension "lon" starting at index 1200. This could degrade performance. Instead, consider rechunking after loading.
  precip_ds = xr.open_dataset(precip, chunks={'time': -1, 'lat': self.chunk_lat, 'lon': self.chunk_lon})
2026-01-28 18:45:01 | INFO     | chunked | Opening PET data
C:\Users\benny\OneDrive\Documents\Github\precip-index\notebook\../src\chunked.py:645: UserWarning: The specified chunks separate the stored chunks along dimension "lon" starting at index 1200. This could degrade performance. Instead, consider rechunking after loading.
  pet_ds = xr.open_dataset(pet, chunks={'time': -1, 'lat': self.chunk_lat, 'lon': self.chunk_lon})
2026-01-28 18:45:01 | INFO     | chunked | 
MemoryEstimate(
  Input size: 223.59 GB
  Peak memory needed: 4024.54 GB
  Available

  [ 21.9%] Chunk 7/32 [lat 0:1200, lon 7200:8400]


2026-01-29 08:17:14 | INFO     | compute | Computing index: shape=(804, 1200, 240), scale=12, distribution=pearson3, grid_cells=288,000
2026-01-29 08:17:14 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-01-29 08:17:24 | INFO     | compute | Step 2/3: Computing pearson3 parameters...
2026-01-29 08:18:37 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-01-29 08:22:15 | INFO     | compute | Index computation complete
2026-01-29 09:01:38 | INFO     | chunked | Processing Chunk 9/32 [lat 1200:2400, lon 0:1200] (28.1%)
2026-01-29 09:04:54 | INFO     | compute | Computing index: shape=(804, 1200, 1200), scale=12, distribution=pearson3, grid_cells=1,440,000
2026-01-29 09:04:54 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-01-29 09:05:46 | INFO     | compute | Step 2/3: Computing pearson3 parameters...
2026-01-29 09:07:16 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-01-29 09:07:50 | INFO     | compute |

  [ 31.2%] Chunk 10/32 [lat 1200:2400, lon 1200:2400]


2026-01-29 11:21:12 | INFO     | compute | Computing index: shape=(804, 1200, 1200), scale=12, distribution=pearson3, grid_cells=1,440,000
2026-01-29 11:21:12 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-01-29 11:22:05 | INFO     | compute | Step 2/3: Computing pearson3 parameters...
2026-01-29 11:32:38 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-01-29 12:09:04 | INFO     | compute | Index computation complete
2026-01-29 12:55:08 | INFO     | chunked | Processing Chunk 12/32 [lat 1200:2400, lon 3600:4800] (37.5%)
2026-01-29 12:58:45 | INFO     | compute | Computing index: shape=(804, 1200, 1200), scale=12, distribution=pearson3, grid_cells=1,440,000
2026-01-29 12:58:45 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-01-29 12:59:38 | INFO     | compute | Step 2/3: Computing pearson3 parameters...
2026-01-29 13:16:21 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-01-29 14:15:40 | INFO     | co

...

2026-01-30 10:16:17 | INFO     | compute | Computing index: shape=(804, 720, 1200), scale=12, distribution=pearson3, grid_cells=864,000
2026-01-30 10:16:17 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-01-30 10:16:53 | INFO     | compute | Step 2/3: Computing pearson3 parameters...
2026-01-30 10:25:54 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-01-30 10:54:46 | INFO     | compute | Index computation complete
2026-01-30 11:48:30 | INFO     | chunked | Processing Chunk 28/32 [lat 3600:4320, lon 3600:4800] (87.5%)
2026-01-30 11:49:55 | INFO     | compute | Computing index: shape=(804, 720, 1200), scale=12, distribution=pearson3, grid_cells=864,000
2026-01-30 11:49:55 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-01-30 11:50:28 | INFO     | compute | Step 2/3: Computing pearson3 parameters...
2026-01-30 12:03:20 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-01-30 12:49:12 | INFO     | compute 

  [ 90.6%] Chunk 29/32 [lat 3600:4320, lon 4800:6000]


2026-01-30 15:55:42 | INFO     | compute | Computing index: shape=(804, 720, 1200), scale=12, distribution=pearson3, grid_cells=864,000
2026-01-30 15:55:42 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-01-30 15:56:13 | INFO     | compute | Step 2/3: Computing pearson3 parameters...
2026-01-30 16:13:27 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-01-30 17:12:19 | INFO     | compute | Index computation complete
2026-01-30 18:03:38 | INFO     | chunked | Processing Chunk 31/32 [lat 3600:4320, lon 7200:8400] (96.9%)
2026-01-30 18:04:59 | INFO     | compute | Computing index: shape=(804, 720, 1200), scale=12, distribution=pearson3, grid_cells=864,000
2026-01-30 18:04:59 | INFO     | compute | Step 1/3: Applying temporal scaling...
2026-01-30 18:05:31 | INFO     | compute | Step 2/3: Computing pearson3 parameters...
2026-01-30 18:19:47 | INFO     | compute | Step 3/3: Transforming to standard normal...
2026-01-30 19:14:45 | INFO     | compute 

  [100.0%] Chunk 32/32 [lat 3600:4320, lon 8400:8640]


2026-01-30 21:07:17 | INFO     | indices | Fitting parameters saved: C:\Users\benny\OneDrive\Documents\Github\precip-index\global\output\params\wld_cli_terraclimate_spei_pearson3_12_month_params.nc
2026-01-30 21:07:17 | INFO     | chunked | Chunked SPEI computation complete: C:\Users\benny\OneDrive\Documents\Github\precip-index\global\output\netcdf\wld_cli_terraclimate_spei_pearson3_12_month.nc


### 5.3 Option C: Dask-based Lazy SPEI

In [ ]:
# ============================================================
# Option C: Dask lazy SPEI
# ============================================================
# For SPEI with Dask, compute the water balance (P - PET + offset)
# first, then pass it to compute_index_dask.
# Uncomment to run:

# from config import SPEI_WATER_BALANCE_OFFSET
#
# precip_ds = xr.open_dataset(
#     str(input_dir / tc_precip_file),
#     chunks={'time': -1, 'lat': 1200, 'lon': 1200}
# )
# pet_ds = xr.open_dataset(
#     str(input_dir / tc_pet_file),
#     chunks={'time': -1, 'lat': 1200, 'lon': 1200}
# )
#
# # Compute water balance
# water_balance = (precip_ds['ppt'] - pet_ds['pet']) + SPEI_WATER_BALANCE_OFFSET
#
# # Build lazy computation graph
# spei_lazy, _ = compute_index_dask(
#     water_balance,
#     scale=12,
#     data_start_year=1958,
#     calibration_start_year=calib_start,
#     calibration_end_year=calib_end,
#     periodicity=Periodicity.monthly,
#     distribution='pearson3'
# )
#
# # Stream to disk
# spei_lazy.to_netcdf(str(output_netcdf / 'spei_12_terraclimate_dask.nc'))

print("Option C: Dask lazy SPEI (uncomment to run)")

### 5.4 Reusing SPEI Parameters

Same workflow as SPI --- save parameters once, reload for subsequent runs.

In [ ]:
# Load previously saved SPEI parameters
# params_spei = load_fitting_params(
#     str(output_params / 'wld_cli_terraclimate_spei_pearson3_12_month_params.nc'),
#     scale=12,
#     periodicity='monthly'
# )
# print(f"Loaded SPEI parameters: {list(params_spei.keys())}")
# for k, v in params_spei.items():
#     if hasattr(v, 'shape'):
#         print(f"  {k}: shape={v.shape}")

print("SPEI parameter reuse workflow (uncomment to run)")

## 6. Batch Processing: Multi-Scale & Multi-Distribution

For comprehensive analysis, you may want to compute indices at multiple scales
(e.g., 1, 3, 6, 12 months) and/or with multiple distributions (Gamma, Pearson III,
Log-Logistic). This section shows how to automate these combinations.

Each combination produces a separate output file. Processing time scales linearly
with the number of combinations.

### 6.1 Multi-Scale, Single Distribution

In [ ]:
# ============================================================
# Compute SPI at multiple scales with Gamma distribution
# ============================================================
# Uncomment to run:

# scales = [1, 3, 6, 12]
# distribution = 'gamma'
#
# for scale in scales:
#     print(f"\n{'='*60}")
#     print(f"Processing SPI-{scale} ({distribution})")
#     print(f"{'='*60}")
#
#     processor.compute_spi_chunked(
#         precip=str(input_dir / chirps_file),
#         output_path=str(output_netcdf / f'wld_cli_chirps3_d2_spi_{distribution}_{scale}_month.nc'),
#         scale=scale,
#         periodicity='monthly',
#         calibration_start_year=calib_start,
#         calibration_end_year=calib_end,
#         distribution=distribution,
#         save_params=True,
#         params_path=str(output_params / f'wld_cli_chirps3_d2_spi_{distribution}_{scale}_month_params.nc'),
#         compress=True,
#         complevel=4,
#         callback=progress_callback
#     )
#
# print(f"\nCompleted SPI at scales: {scales}")

print("Multi-scale SPI batch processing (uncomment to run)")

### 6.2 Single Scale, Multiple Distributions

In [ ]:
# ============================================================
# Compute SPI-12 with all supported distributions
# ============================================================
# Uncomment to run:

# scale = 12
# distributions = ['gamma', 'pearson3', 'log_logistic']
#
# for dist in distributions:
#     print(f"\n{'='*60}")
#     print(f"Processing SPI-{scale} ({dist})")
#     print(f"{'='*60}")
#
#     processor.compute_spi_chunked(
#         precip=str(input_dir / chirps_file),
#         output_path=str(output_netcdf / f'wld_cli_chirps3_d2_spi_{dist}_{scale}_month.nc'),
#         scale=scale,
#         periodicity='monthly',
#         calibration_start_year=calib_start,
#         calibration_end_year=calib_end,
#         distribution=dist,
#         save_params=True,
#         params_path=str(output_params / f'wld_cli_chirps3_d2_spi_{dist}_{scale}_month_params.nc'),
#         compress=True,
#         complevel=4,
#         callback=progress_callback
#     )
#
# print(f"\nCompleted SPI-{scale} with distributions: {distributions}")

print("Multi-distribution SPI batch processing (uncomment to run)")

### 6.3 Multi-Scale, Multiple Distributions

In [ ]:
# ============================================================
# Full combination: multiple scales x multiple distributions
# ============================================================
# WARNING: This produces many output files and can take a long time.
# For example, 4 scales x 3 distributions = 12 separate runs.
# Uncomment to run:

# scales = [3, 6, 12]
# distributions = ['gamma', 'pearson3', 'log_logistic']
#
# total = len(scales) * len(distributions)
# run = 0
#
# for scale in scales:
#     for dist in distributions:
#         run += 1
#         print(f"\n{'='*60}")
#         print(f"[{run}/{total}] SPI-{scale} ({dist})")
#         print(f"{'='*60}")
#
#         processor.compute_spi_chunked(
#             precip=str(input_dir / chirps_file),
#             output_path=str(output_netcdf / f'wld_cli_chirps3_d2_spi_{dist}_{scale}_month.nc'),
#             scale=scale,
#             periodicity='monthly',
#             calibration_start_year=calib_start,
#             calibration_end_year=calib_end,
#             distribution=dist,
#             save_params=True,
#             params_path=str(output_params / f'wld_cli_chirps3_d2_spi_{dist}_{scale}_month_params.nc'),
#             compress=True,
#             complevel=4,
#             callback=progress_callback
#         )
#
# print(f"\nCompleted all {total} combinations")

print("Full batch processing (uncomment to run)")

### 6.4 Batch SPEI (Multi-Scale / Multi-Distribution)

The same loop patterns work for SPEI --- just replace `compute_spi_chunked` with
`compute_spei_chunked` and provide both precipitation and PET files.

In [ ]:
# ============================================================
# Batch SPEI: multiple scales with Pearson III
# ============================================================
# Uncomment to run:

# scales = [3, 6, 12]
# distribution = 'pearson3'
#
# for scale in scales:
#     print(f"\n{'='*60}")
#     print(f"Processing SPEI-{scale} ({distribution})")
#     print(f"{'='*60}")
#
#     processor.compute_spei_chunked(
#         precip=str(input_dir / tc_precip_file),
#         pet=str(input_dir / tc_pet_file),
#         output_path=str(output_netcdf / f'wld_cli_terraclimate_spei_{distribution}_{scale}_month.nc'),
#         scale=scale,
#         periodicity='monthly',
#         calibration_start_year=calib_start,
#         calibration_end_year=calib_end,
#         distribution=distribution,
#         save_params=True,
#         params_path=str(output_params / f'wld_cli_terraclimate_spei_{distribution}_{scale}_month_params.nc'),
#         compress=True,
#         complevel=4,
#         callback=progress_callback
#     )
#
# print(f"\nCompleted SPEI at scales: {scales}")

print("Batch SPEI processing (uncomment to run)")

## 7. Performance Tips

1. **Use SSD storage** --- Significantly faster I/O for chunked processing
2. **Pre-process data** --- Ensure data is already in (time, lat, lon) order
3. **Compress output** --- Use `compress=True` to reduce file sizes by 50--70%
4. **Monitor memory** --- Watch for memory leaks during long runs
5. **Use Zarr for large outputs** --- Better for parallel writes than NetCDF
6. **Choose the right method:**
   - **Option A** (Simple) --- Best for most users, minimal code
   - **Option B** (Chunked) --- When you need progress tracking or custom chunks
   - **Option C** (Dask) --- When data doesn't fit even with chunking
7. **Reuse parameters** --- After first run with `save_params=True`, subsequent runs skip the expensive fitting step
8. **Distribution defaults** --- Gamma uses Method of Moments; Pearson III uses Method of Moments; Log-Logistic uses MLE

In [ ]:
# Monitor memory usage during processing
print_memory_info()

## 8. Summary

### Decision Flowchart

```
Does your data fit in RAM?
├── YES → Use standard spi() / spei() from indices module
│         (See notebooks 01-02)
│
└── NO → How much control do you need?
         ├── Minimal → compute_spi_global() / compute_spei_global()
         │             (Section 4.1 / 5.1)
         │
         ├── Full control → ChunkedProcessor
         │                  (Section 4.2 / 5.2)
         │
         └── Very large → compute_index_dask / Zarr
                          (Section 4.3 / 5.3)
```

### Available Distributions

| Distribution | Default Method | Best For |
|:------------|:--------------|:---------|
| Gamma | Method of Moments | SPI (standard choice) |
| Pearson III | Method of Moments | SPEI, skewed data |
| Log-Logistic | MLE | Better tail behavior |

### Next Steps

- **Notebook 01**: Calculate SPI with multiple distributions
- **Notebook 02**: Calculate SPEI with PET options
- **Notebook 03**: Analyze climate extreme events using run theory
- **Notebook 04**: Explore the visualization gallery